# Parte III: Transformación y Análisis Avanzado de Datos con Pandas

En esta tercera parte del proyecto, continuaremos trabajando con el dataset de ventas que utilizamos en la Parte II. En esta fase, aplicaremos técnicas avanzadas de transformación y análisis de datos utilizando las nuevas habilidades adquiridas en Pandas, tales como agrupaciones complejas y el uso del método apply. Nos enfocaremos en extraer insights más profundos y preparar los datos para futuros análisis y modelos predictivos.



In [1]:
import numpy as np
import pandas as pd

df=pd.read_csv("../data/retail_sales_dataset.csv")

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    1000 non-null   int64 
 1   Date              1000 non-null   object
 2   Customer ID       1000 non-null   object
 3   Gender            1000 non-null   object
 4   Age               1000 non-null   int64 
 5   Product Category  1000 non-null   object
 6   Quantity          1000 non-null   int64 
 7   Price per Unit    1000 non-null   int64 
 8   Total Amount      1000 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 70.4+ KB


In [3]:
df.sample(10)

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
886,887,2023-06-11,CUST887,Male,59,Clothing,4,25,100
700,701,2023-12-14,CUST701,Female,52,Beauty,2,30,60
770,771,2023-12-13,CUST771,Male,24,Electronics,2,25,50
322,323,2023-01-26,CUST323,Female,29,Beauty,3,300,900
659,660,2023-04-29,CUST660,Female,38,Beauty,2,500,1000
817,818,2023-05-18,CUST818,Male,30,Electronics,1,500,500
987,988,2023-05-28,CUST988,Female,63,Clothing,3,25,75
643,644,2023-09-06,CUST644,Male,23,Beauty,3,25,75
179,180,2023-01-01,CUST180,Male,41,Clothing,3,300,900
823,824,2023-05-05,CUST824,Male,63,Clothing,4,30,120


In [4]:
#cambiar columna date a datetime
df['Date'] = pd.to_datetime(df['Date'])

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    1000 non-null   int64         
 1   Date              1000 non-null   datetime64[ns]
 2   Customer ID       1000 non-null   object        
 3   Gender            1000 non-null   object        
 4   Age               1000 non-null   int64         
 5   Product Category  1000 non-null   object        
 6   Quantity          1000 non-null   int64         
 7   Price per Unit    1000 non-null   int64         
 8   Total Amount      1000 non-null   int64         
dtypes: datetime64[ns](1), int64(5), object(3)
memory usage: 70.4+ KB


In [6]:
# Extraer el mes de la columna Date y crear una nueva columna
df['Mes'] = df['Date'].dt.month

In [7]:
# Crear una nueva columna con el nombre completo del mes
df['nombreMes'] = df['Date'].dt.month_name()

In [8]:
df.describe(include='all')

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,Mes,nombreMes
count,1000.000000,1000,1000,1000,1000.00000,1000,1000.000000,1000.000000,1000.000000,1000.000000,1000
unique,NaN,NaN,1000,2,NaN,3,NaN,NaN,NaN,NaN,12
top,NaN,NaN,CUST1000,Female,NaN,Clothing,NaN,NaN,NaN,NaN,May
freq,NaN,NaN,1,510,NaN,351,NaN,NaN,NaN,NaN,105
mean,500.500000,2023-07-03 00:25:55.200000256,NaN,NaN,41.39200,NaN,2.514000,179.890000,456.000000,6.549000,NaN
min,1.000000,2023-01-01 00:00:00,NaN,NaN,18.00000,NaN,1.000000,25.000000,25.000000,1.000000,NaN
25%,250.750000,2023-04-08 00:00:00,NaN,NaN,29.00000,NaN,1.000000,30.000000,60.000000,4.000000,NaN
50%,500.500000,2023-06-29 12:00:00,NaN,NaN,42.00000,NaN,3.000000,50.000000,135.000000,6.000000,NaN
75%,750.250000,2023-10-04 00:00:00,NaN,NaN,53.00000,NaN,4.000000,300.000000,900.000000,10.000000,NaN
max,1000.000000,2024-01-01 00:00:00,NaN,NaN,64.00000,NaN,4.000000,500.000000,2000.000000,12.000000,NaN


## Transformación de Datos
* Crea nuevas columnas: Basándonos en los datos existentes, crea nuevas columnas que sean útiles para el análisis. Por ejemplo, calcula el ingreso total por venta y normaliza las ventas.
* Clasifica los datos: Crea una columna que clasifique las ventas en categorías significativas (e.g., ‘Alta’, ‘Media’, ‘Baja’).


In [9]:
#el ingreso total por venta, se encuentra en la tabla.

#de acuerdo con Gemini, normalizar las ventas, para machine learning podria referirse a escalamiento
#asi que me dio la formula para lograr el valor normalizado: x-min/(max-min)

def NormalizCol(columna):
    valores_min = columna.min()
    valores_max = columna.max()
    return (columna - valores_min) / (valores_max - valores_min)

# Aplicarlo a la columna 'total'
df['NormTotalAmount'] = NormalizCol(df['Total Amount'])

In [10]:
#Para obtener la categoria de ventas alta media y baja, vamos a sacar tercios de la compra total, segun cuartiles
limiteBaja = df['Total Amount'].quantile(0.33)
limiteAlta = df['Total Amount'].quantile(0.67)
print(limiteBaja, limiteAlta)

90.0 500.0


In [11]:
#buscando el google, me enteré  del metodo qcut y no necesite usar los limites anteriores
df['TipoVenta'] = pd.qcut(df['Total Amount'], q=3, labels=['Baja', 'Media', 'Alta'])


In [12]:
df.sample(10)

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,Mes,nombreMes,NormTotalAmount,TipoVenta
860,861,2023-02-17,CUST861,Female,41,Clothing,3,30,90,2,February,0.032911,Baja
198,199,2023-12-04,CUST199,Male,45,Beauty,3,500,1500,12,December,0.746835,Alta
603,604,2023-09-11,CUST604,Female,29,Electronics,4,50,200,9,September,0.088608,Media
13,14,2023-01-17,CUST014,Male,64,Clothing,4,30,120,1,January,0.048101,Media
965,966,2023-02-20,CUST966,Male,60,Electronics,2,500,1000,2,February,0.493671,Alta
179,180,2023-01-01,CUST180,Male,41,Clothing,3,300,900,1,January,0.443038,Alta
635,636,2023-03-23,CUST636,Female,21,Beauty,3,500,1500,3,March,0.746835,Alta
554,555,2023-10-19,CUST555,Male,25,Beauty,1,300,300,10,October,0.139241,Media
476,477,2023-04-24,CUST477,Male,43,Clothing,4,30,120,4,April,0.048101,Media
170,171,2023-11-24,CUST171,Female,52,Clothing,3,300,900,11,November,0.443038,Alta



## Agrupación y Agregación
* Agrupación por múltiples columnas: Realiza agrupaciones por categorías como producto y tienda, producto y mes, etc.
* Aplicar funciones de agregación: Utiliza funciones como sum, mean, count, min, max, std, y var para obtener estadísticas descriptivas de cada grupo.

In [13]:
#REVISAR CUANTO GASTO CADA GENERO EN LAS COMPRAS DEL DATASET
df.groupby('Gender')['Total Amount'].sum()

Gender
Female    232840
Male      223160
Name: Total Amount, dtype: int64

In [14]:
#REVISAR LOS INGRESOS POR CATEGORIA
df.groupby('Product Category')['Total Amount'].sum()

Product Category
Beauty         143515
Clothing       155580
Electronics    156905
Name: Total Amount, dtype: int64

---
* No se observan grandes diferencias en el ingreso por genero. En total las mujeres gastaron cerca de 10 mil dolares mas que los hombres en todo un año.
* En categoria, tanto Ropa como Electronica generaron ingresos similares, aunque los productos de belleza solo se quedaron atras por unos 12-15 mil dolares anuales
---

In [15]:
#REVISAR LAS COMPRAS MENSUALES
df.groupby('nombreMes')['Total Amount'].sum()

nombreMes
April        33870
August       36960
December     44690
February     44060
January      36980
July         35465
June         36715
March        28990
May          53150
November     34920
October      46580
September    23620
Name: Total Amount, dtype: int64

In [16]:
#REVISAR LAS COMPRAS MENSUALES, PERO JUNTANDO EL NUMERO DEL MES CON EL NOMBRE PARA VERLO DE FORMA TEMPORAL
df.groupby(['Mes', 'nombreMes']).agg(
    Total=('Total Amount', 'sum'),
    Total_Normalizado=('NormTotalAmount', 'sum'),
    Promedio=('Total Amount', 'mean')).round(2)

,,Total,Total_Normalizado,Promedio
Mes,nombreMes,,,
1,January,36980,17.74,474.10
2,February,44060,21.23,518.35
3,March,28990,13.75,397.12
4,April,33870,16.06,393.84
5,May,53150,25.58,506.19
6,June,36715,17.62,476.82
7,July,35465,17.05,492.57
8,August,36960,17.52,393.19
9,September,23620,11.14,363.38


---
* Se observa la mayoria de las compras ocurrio en los meses de Mayo, Octubre y Diciembre
* Septiembre fue el mes con menor ventas
---

In [17]:
df.groupby(['TipoVenta']).agg(
    Total=('Total Amount', 'sum'),
    Total_Normalizado=('NormTotalAmount', 'sum'),
    Promedio=('Total Amount', 'mean')).round(2)

C:\Users\jr_ca\AppData\Local\Temp\ipykernel_31408\2910738268.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['TipoVenta']).agg(


,Total,Total_Normalizado,Promedio
TipoVenta,,,
Baja,18440,4.92,52.84
Media,73960,32.99,210.11
Alta,363600,180.32,1216.05


---
* Los ingresos por compras "ALTAS" corresponden a 5 veces el tipo de compras Media y casi 20 veces las compras Baja
* Los ingresos por compras "MEDIAS" fueron 4 veces el tipo de compras Baja 
---

In [18]:
df.groupby(['TipoVenta', 'Mes']).agg(
    Total=('Total Amount', 'sum'),
    Promedio=('Total Amount', 'mean')).round(2)

C:\Users\jr_ca\AppData\Local\Temp\ipykernel_31408\4037493369.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['TipoVenta', 'Mes']).agg(


Total  Promedio
TipoVenta Mes                 
Baja      1     1750     56.45
          2     1220     43.57
          3      860     53.75
          4     2000     51.28
          5     2200     55.00
          6     1245     47.88
          7     1175     58.75
          8     1810     50.28
          9     1280     58.18
          10    1900     57.58
          11    1560     55.71
          12    1440     48.00
Media     1     4930    224.09
          2     5240    187.14
          3     8030    205.90
          4     4770    190.80
          5     7150    246.55
          6     5670    218.08
          7     6990    233.00
          8     8450    234.72
          9     5240    180.69
          10    6880    202.35
          11    4960    183.70
          12    5650    209.26
Alta      1    30300   1212.00
          2    37600   1296.55
          3    20100   1116.67
          4    27100   1231.82
          5    43800   1216.67
          6    29800   1192.00
          7    27300   1240.91
          8    26700   1213.64
          9    17100   1221.43
          10   37800   1303.45
          11   28400   1234.78
          12   37600   1105.88

---
* Las compras bajas tuvieron su mejor mes en Mayo
* Las compras medias en Agosto y Marzo
* Las compras altas generaron mas ingresos en Mayo y Octubre

* Con la informacion anterior, de mejores vental total en Mayo, Octubre y Diciembre, vemos la influencia de las compras Altas en el total anual
---


## Análisis Personalizado con apply
* Función personalizada: Aplica funciones personalizadas para realizar análisis específicos que no se pueden lograr con las funciones de agregación estándar.
* Ejemplo de uso avanzado: Calcula la desviación de cada venta respecto a la media de su grupo.


In [ ]:
#se calcula el promedio por categoria alta, media, baja
promedios = df.groupby('TipoVenta')['Total Amount'].mean()
promedios

C:\Users\jr_ca\AppData\Local\Temp\ipykernel_31408\3600593377.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  promedios = df.groupby('TipoVenta')['Total Amount'].mean()


TipoVenta
Baja       52.836676
Media     210.113636
Alta     1216.053512
Name: Total Amount, dtype: float64

In [ ]:
#creamos funcion para calcular desviacion
#idea salio desde: https://blog.finxter.com/calculating-mean-absolute-deviation-in-dataframe-rows-and-columns-using-python/
#manteniendo la solicitud que fuese usando apply

def desviacion(fila):
    return fila['Total Amount'] - promedios[fila['TipoVenta']]

#Se crea nueva columna aplicando la funcion anterior
df['Desviacion'] = df.apply(desviacion, axis=1)

In [21]:
df

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,Mes,nombreMes,NormTotalAmount,TipoVenta,Desviacion
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,11,November,0.063291,Media,-60.113636
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,2,February,0.493671,Alta,-216.053512
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,1,January,0.002532,Baja,-22.836676
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,5,May,0.240506,Media,289.886364
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,5,May,0.037975,Media,-110.113636
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,2023-05-16,CUST996,Male,62,Clothing,1,50,50,5,May,0.012658,Baja,-2.836676
996,997,2023-11-17,CUST997,Male,52,Beauty,3,30,90,11,November,0.032911,Baja,37.163324
997,998,2023-10-29,CUST998,Female,23,Beauty,4,25,100,10,October,0.037975,Media,-110.113636
998,999,2023-12-05,CUST999,Female,36,Electronics,3,50,150,12,December,0.063291,Media,-60.113636
